## Purpose:
Derive analysis-friendly features from the cleaned data: a reporting-timeliness category, a quality band, a discrepancy severity tier, a discrepancy-type count, and a visit month for trend analysis.


In [1]:
import pandas as pd
from config import CLEAN_PATH, FEATURED_PATH


In [2]:
df = pd.read_csv(CLEAN_PATH, parse_dates=["visit_date", "submission_time"])


In [3]:
def reporting_status(row):
    if row["report_submitted"].strip().lower() == "yes":
        return "On time"
    elif row["days_late"] <= 7:
        return "Late 1-7 days"
    else:
        return "Late 8+ days"
        

In [4]:
df["reporting_status"] = df.apply(reporting_status, axis=1)

def quality_band(score):
    if score <= 2:
        return "Poor (1-2)"
    elif score == 3:
        return "Fair (3)"
    else:
        return "Good (4-5)"


In [5]:
df["quality_band"] = df["quality_score"].apply(quality_band)


In [6]:
def discrepancy_severity(count):
    if count == 0:
        return "None"
    elif count <= 10:
        return "Low"
    elif count <= 25:
        return "Medium"
    else:
        return "High"

In [7]:
df["discrepancy_severity"] = df["discrepancy_count"].apply(discrepancy_severity)

disc_cols = ["disc_missing_entries", "disc_miscounted_totals",
             "disc_illegible_records", "disc_wrong_indicator"]
df["num_discrepancy_types"] = df[disc_cols].sum(axis=1)

df["visit_month"] = df["visit_date"].dt.to_period("M").astype(str)


In [8]:
print(df[["reporting_status","quality_band","discrepancy_severity","num_discrepancy_types","visit_month"]].head(10))
print()
print(df["reporting_status"].value_counts())
print(df["quality_band"].value_counts())
print(df["discrepancy_severity"].value_counts())
print(df["visit_month"].value_counts())

  reporting_status quality_band discrepancy_severity  num_discrepancy_types  \
0          On time   Good (4-5)                  Low                      1   
1          On time   Poor (1-2)               Medium                      1   
2     Late 8+ days   Poor (1-2)                  Low                      2   
3          On time   Good (4-5)               Medium                      0   
4     Late 8+ days   Poor (1-2)                  Low                      3   
5          On time   Poor (1-2)               Medium                      4   
6          On time   Poor (1-2)                  Low                      2   
7          On time   Good (4-5)                  Low                      2   
8     Late 8+ days     Fair (3)                 None                      0   
9          On time   Poor (1-2)                  Low                      1   

  visit_month  
0     2026-08  
1     2026-08  
2     2026-07  
3     2026-08  
4     2026-06  
5     2026-06  
6     2026-07  
7 

In [9]:
df.to_csv(FEATURED_PATH, index=False)
print("\nSaved:", FEATURED_PATH)
 


Saved: .\facility_dq_featured.csv
